In [2]:
import torch
import torch.nn as nn
from einops import rearrange, einsum
from einops.layers.torch import Rearrange


class Attention(nn.Module):
    def __init__(
            self,
            dim,
            heads = 8,
            kv_heads = 1, # How much kv heads are there?
    ):
        super(Attention, self).__init__()

        assert kv_heads <= heads and (heads % kv_heads == 0), "GQA하시려면 KV 헤드의 갯수가 query 헤드의 수의 약수여야 합니다."

        self.heads = heads
        self.kv_heads = kv_heads

        self.dim_head = dim // heads # Query head dim
        self.dim_kv = self.dim_head * kv_heads # dimensions we need for all kv heads(before splitting each)
        self.num_grouped_queries = heads // kv_heads # How many queries attend to one kv head

        self.scale = self.dim_head ** -0.5 # for attend
        self.norm = nn.RMSNorm(dim)

        # Batch, Head, Number(Sequence length), Dimension 형태로 쭈욱 활용한다. 처음 Q,K,V 분리는 B, N, D.
        # Batch는 항상 살려두고, Head끼리 적절히 쪼개 배분해 최종 concat해 반환하는 방법으로 GQA-supported attend() 함수를 만들어야 한다.

        self.qkv_split = (dim, self.dim_kv, self.dim_kv) # q, k, v
        self.to_qkv = nn.Linear(dim, sum(self.qkv_split), bias=False) # 한번에 Q, K, V 다 구해서, qkv_split으로 쪼개야 함. head로 쪼개는건 einops Rearrange 활용.

        self.split_heads = Rearrange('b n (h d) -> b h n d', d = self.dim_head)

    def forward(self, x):
        q, k, v = self.to_qkv(x).split(self.qkv_split, dim = -1)
        q, k, v = map(self.split_heads, (q, k, v))

        q = rearrange(q, 'b (h qh) ... -> b h qh ...', qh = self.num_grouped_queries)
        print(q.shape)
        sim = einsum(q, k, 'b h qh i d, b h j d -> b h qh i j') * self.scale
        print(sim.shape)
        attn = sim.softmax(dim=-1)
        print(torch.sum(attn.squeeze()[0, 0], dim=-1))
        attn_out = einsum(attn, v, 'b h qh i j, b h j d -> b h qh i d')
        print(attn_out.shape)
        attn_out = rearrange(attn_out, 'b h qh ... -> b (h qh) ...')
        print(attn_out.shape)

        return attn_out


In [15]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, hidden_dim, num_attn_heads, num_kv_heads):
        super(GroupedQueryAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_attn_heads = num_attn_heads
        self.num_kv_heads = num_kv_heads

        assert hidden_dim % num_attn_heads == 0 and hidden_dim % num_kv_heads == 0
        assert num_attn_heads > 0 and num_kv_heads > 0
        assert num_attn_heads >= num_kv_heads and num_attn_heads % num_kv_heads == 0

        self.head_dim = hidden_dim // num_attn_heads

        self.q_proj = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.k_proj = nn.Linear(self.hidden_dim, self.head_dim * num_kv_heads)
        self.v_proj = nn.Linear(self.hidden_dim, self.head_dim * num_kv_heads)
        self.out_proj = nn.Linear(self.hidden_dim, self.hidden_dim)

    def forward(self, seq):
        batch_size, seq_len, _ = seq.size()

        q = self.q_proj(seq)
        k = self.k_proj(seq)
        v = self.v_proj(seq)

        q = q.view(batch_size, seq_len, self.num_attn_heads, self.head_dim).permute(0, 2, 1, 3) # B N H D -> B H N D
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).permute(0, 2, 1, 3)

        k = torch.repeat_interleave(k, self.num_attn_heads // self.num_kv_heads, dim=1)
        v = torch.repeat_interleave(v, self.num_attn_heads // self.num_kv_heads, dim=1)

        attn_score = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_score = torch.softmax(attn_score, dim=-1)
        attn_score = torch.matmul(attn_score, v)
        print(attn_score.permute(0,2,1,3).shape)
        attn_score = attn_score.permute(0, 2, 1, 3).reshape(batch_size, seq_len, self.hidden_dim)

        attn_output = self.out_proj(attn_score)
        return attn_output

In [16]:
X = torch.randn(1, 32, 16)
attend = Attention(16, 8, 8)
print(attend(X).shape)

torch.Size([1, 32, 8, 2])
torch.Size([1, 32, 16])


In [5]:
X = torch.randn(1, 32, 16) # b, sequence_length, embedding dimension
attend = Attention(16, heads=8, kv_heads=1)
print(attend(X).shape)


torch.Size([1, 1, 8, 32, 2])
torch.Size([1, 1, 8, 32, 32])
tensor(1.0000, grad_fn=<SumBackward1>)
torch.Size([1, 1, 8, 32, 2])
torch.Size([1, 8, 32, 2])
torch.Size([1, 8, 32, 2])
